# product-slm: QLoRA fine-tune on a free T4 (Colab or Kaggle)

The notebook runs these steps:
1. Install the repo with the `train` extra and `bitsandbytes`.
2. Run QLoRA on **Qwen2.5-1.5B-Instruct** over the 600 distilled train labels. This takes ~10–20 min on a T4.
3. Evaluate on val and test with the same runner and metrics as every other model in the repo. It also runs the **zero-shot base model** with the same short prompt, so the fine-tuning gain is measured, not assumed.
4. Package everything for download:
   - the adapter;
   - the predictions;
   - the cassettes, which let CI replay these exact outputs without a GPU;
   - the training summary.

**Runtime → Change runtime type → T4 GPU** before running anything.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/houssem019/product-slm.git
%cd product-slm
!pip install -q -e ".[train]" bitsandbytes

## 1. Build the SFT data
The SFT data is chat triples (short student prompt, title, canonical JSON). The test split is never exported for training.

In [ ]:
!python -m pslm.train.format

## 2. QLoRA training
These settings come from `pslm/train/train.py`:
- **Base model:** 4-bit NF4 with double quantization, fp16 compute (a T4 has no bf16).
- **LoRA:** r=16, alpha=32 on all attention and MLP projections.
- **Schedule:** 3 epochs, lr 2e-4 with cosine decay, effective batch 16, gradient checkpointing.
- **Loss:** answer tokens only.

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
OUT = "outputs/qwen2.5-1.5b-lora"
!python -m pslm.train.train --base {BASE} --load-in-4bit --out {OUT}

In [ ]:
import json
from pathlib import Path

s = json.loads(Path(f"{OUT}/train_summary.json").read_text())
print(
    {k: s[k] for k in ("train_loss", "eval_loss", "runtime_s", "device", "trainable_params", "total_params")}
)
for e in s["log_history"]:
    if "eval_loss" in e:
        print(f"epoch {e['epoch']:.0f}: eval_loss {e['eval_loss']:.4f}")

## 3. Evaluate: fine-tuned vs zero-shot base, on val and test
- **Decoding:** greedy and unconstrained, so JSON validity is the model's own. vLLM guided decoding comes at serving time.
- **`--record`:** writes cassettes, so CI can replay these runs on a CPU-only runner.

In [ ]:
!python -m pslm.distill.label val --provider hf --model {OUT}/adapter --name qwen2.5-1.5b-lora --record
!python -m pslm.distill.label test --provider hf --model {OUT}/adapter --name qwen2.5-1.5b-lora --record
!python -m pslm.distill.label val --provider hf --model {BASE} --name qwen2.5-1.5b-zeroshot --prompt student --record
!python -m pslm.distill.label test --provider hf --model {BASE} --name qwen2.5-1.5b-zeroshot --prompt student --record

## 4. Download the artifacts
Unzip the archive at the repo root locally and commit it. `outputs/` stays gitignored; the adapter goes to the Hub, or stays local.

In [ ]:
!zip -qr slm_results.zip data/preds data/cassettes {OUT}/train_summary.json {OUT}/adapter

In [ ]:
from google.colab import files  # on Kaggle: download slm_results.zip from the Output tab instead

files.download("slm_results.zip")

Optional: publish the adapter on the Hugging Face Hub.
1. Run `from huggingface_hub import login; login()` with a write token.
2. Add `--push-to-hub houssem019/product-slm-qwen2.5-1.5b-lora` to the training command in step 2.